In [2]:
"""
MScFE 610 Group Work Project #2 - Complete Technical Analysis
Challenges: Non-Stationarity (ECM), Regime Change, Feature Extraction (PCA)

Authors: Anas Ebaid, Jun Hyung Lee, Utman Rasheed
Date: January 2026


# ==========================================
# CHALLENGE 1: MODELING NON-STATIONARITY (ECM)
# ==========================================
print("\n" + "="*70)
print("CHALLENGE 1: ERROR-CORRECTION MODEL (ECM)")
print("="*70)

# 1. Load Data (US GDP & Consumption, 1959-1992)
print("\n[1/6] Loading US Macroeconomic Data...")
data = sm.datasets.macrodata.load_pandas().data
data = data.assign(date=pd.date_range("1959-01-01", periods=len(data), freq="QS"))
data.set_index("date", inplace=True)
Y = data["realcons"]  # Real consumption
X = data["realgdp"]   # Real GDP
print(f"   Data shape: {data.shape}, Period: {data.index[0].date()} to {data.index[-1].date()}")

# 2. Unit Root Tests (ADF)
print("\n[2/6] Unit Root Tests (Augmented Dickey-Fuller)...")
adf_Y_level = adfuller(Y.dropna(), regression='ct', autolag='AIC')
adf_X_level = adfuller(X.dropna(), regression='ct', autolag='AIC')
adf_dY = adfuller(Y.diff().dropna(), regression='c', autolag='AIC')
adf_dX = adfuller(X.diff().dropna(), regression='c', autolag='AIC')

print(f"\n   {'Series':<20} {'ADF Stat':<12} {'p-value':<12} {'Conclusion'}")
print(f"   {'-'*60}")
print(f"   {'Consumption (level)':<20} {adf_Y_level[0]:<12.4f} {adf_Y_level[1]:<12.4f} {'Unit Root (I(1))'}")
print(f"   {'GDP (level)':<20} {adf_X_level[0]:<12.4f} {adf_X_level[1]:<12.4f} {'Unit Root (I(1))'}")
print(f"   {'ΔConsumption':<20} {adf_dY[0]:<12.4f} {adf_dY[1]:<12.4f} {'Stationary (I(0))'}")
print(f"   {'ΔGDP':<20} {adf_dX[0]:<12.4f} {adf_dX[1]:<12.4f} {'Stationary (I(0))'}")

# 3. Cointegration Test (Johansen)
print("\n[3/6] Johansen Cointegration Test...")
jres = coint_johansen(data[["realcons", "realgdp"]], det_order=0, k_ar_diff=1)
print(f"\n   {'Hypothesis':<20} {'Trace Stat':<15} {'Crit. Value (5%)':<18} {'Conclusion'}")
print(f"   {'-'*65}")
print(f"   {'r = 0 (no coint.)':<20} {jres.lr1[0]:<15.4f} {jres.cvt[0, 1]:<18.4f} {'Reject → Cointegration exists' if jres.lr1[0] > jres.cvt[0, 1] else 'Cannot reject'}")
print(f"   {'r ≤ 1':<20} {jres.lr1[1]:<15.4f} {jres.cvt[1, 1]:<18.4f} {'Cannot reject'}")

# 4. Long-Run Equilibrium Regression
print("\n[4/6] Estimating Long-Run Equilibrium (Engle-Granger Step 1)...")
ols = sm.OLS(Y, sm.add_constant(X)).fit()
data["u_hat"] = ols.resid
print(f"\n   Long-run equation: Cons = {ols.params[0]:.2f} + {ols.params[1]:.4f} × GDP")
print(f"   (Interpretation: Marginal propensity to consume ≈ {ols.params[1]:.1%})")

# Test stationarity of equilibrium error
adf_u = adfuller(data["u_hat"].dropna(), regression='c', autolag='AIC')
print(f"   ADF test on equilibrium error: stat={adf_u[0]:.4f}, p-value={adf_u[1]:.4f}")
print(f"   → Equilibrium error is {'stationary (I(0)) ✓' if adf_u[1] < 0.05 else 'NON-stationary (warning!)'}")

# 5. Error-Correction Model (ECM)
print("\n[5/6] Estimating Error-Correction Model (Engle-Granger Step 2)...")
data["dY"] = Y.diff()
data["dX"] = X.diff()
data["u_hat_lag"] = data["u_hat"].shift(1)
ecm_data = data.dropna(subset=["dY", "dX", "u_hat_lag"])

ecm = sm.OLS(ecm_data["dY"], sm.add_constant(ecm_data[["dX", "u_hat_lag"]])).fit()
print(f"\n   ECM equation: ΔCons = {ecm.params[0]:.2f} + {ecm.params[1]:.4f} × ΔGDP + {ecm.params[2]:.4f} × u_{{t-1}}")
print(f"\n   Short-run multiplier (φ): {ecm.params[1]:.4f} (p={ecm.pvalues[1]:.4f})")
print(f"   Adjustment speed (γ):     {ecm.params[2]:.4f} (p={ecm.pvalues[2]:.4f})")
print(f"   R²: {ecm.rsquared:.4f}")
print(f"\n   Interpretation:")
print(f"   • 1% increase in GDP growth → {ecm.params[1]:.2%} increase in consumption growth")
print(f"   • Equilibrium error corrects at {-ecm.params[2]:.1%} per quarter")
print(f"   • Half-life of disequilibrium: {np.log(0.5)/np.log(1+ecm.params[2]):.1f} quarters")

# 6. Diagnostic Plots
print("\n[6/6] Generating Diagnostic Plots...")
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Panel A: Time series with equilibrium
axes[0].plot(data.index, Y, label='Actual Consumption', linewidth=1.5, alpha=0.8)
axes[0].plot(data.index, ols.fittedvalues, label='Long-run Equilibrium',
             linestyle='--', linewidth=2, color='red', alpha=0.7)
axes[0].set_title('Challenge 1A: Consumption vs. Long-Run Equilibrium', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Real Consumption')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Panel B: Equilibrium error (should be stationary)
axes[1].plot(data.index, data['u_hat'], label='Equilibrium Error', linewidth=1, color='purple')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].fill_between(data.index, 0, data['u_hat'], alpha=0.2, color='purple')
axes[1].set_title('Challenge 1B: Error-Correction Term (should fluctuate around zero)',
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('u_t (Residual)')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Panel C: ECM residuals (should be white noise)
ecm_resid = ecm.resid
axes[2].scatter(ecm_data.index, ecm_resid, s=10, alpha=0.5, color='green')
axes[2].axhline(0, color='black', linestyle='--', linewidth=1)
axes[2].set_title('Challenge 1C: ECM Residuals (should be randomly distributed)',
                  fontsize=12, fontweight='bold')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Residual')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('challenge1_ecm_diagnostics.png', dpi=300, bbox_inches='tight')
print("   ✓ Saved: challenge1_ecm_diagnostics.png")
plt.show()

# ==========================================
# CHALLENGE 2: REGIME CHANGE (MARKOV SWITCHING)
# ==========================================
print("\n" + "="*70)
print("CHALLENGE 2: MARKOV-SWITCHING REGIME DETECTION")
print("="*70)

# 1. Load Data (Fed Funds Rate, 1954-2010)
print("\n[1/5] Loading Federal Funds Rate Data...")
fed = pd.Series(fedfunds, index=pd.date_range("1954-07-01", periods=len(fedfunds), freq="QS"))
fed.name = "fedfunds_rate"
print(f"   Data shape: {fed.shape}, Period: {fed.index[0].date()} to {fed.index[-1].date()}")
print(f"   Mean: {fed.mean():.2f}%, Std: {fed.std():.2f}%")

# 2. Fit Markov-Switching Model
print("\n[2/5] Fitting 2-Regime Markov-Switching Model...")
print("   (This may take 30-60 seconds...)")
mod = MarkovRegression(fed, k_regimes=2, switching_variance=True)
res = mod.fit(maxiter=200, disp=False)

# 3. Extract Results
print("\n[3/5] Model Estimation Results:")
regime_means = res.params[[0, 1]]
regime_vars = res.params[[2, 3]]
P = res.transition_matrix

print(f"\n   {'Regime':<10} {'Mean (%)':<12} {'Variance':<12} {'Interpretation'}")
print(f"   {'-'*60}")
print(f"   {'Regime 0':<10} {regime_means[0]:<12.4f} {regime_vars[0]:<12.4f} {'Low-Rate Environment'}")
print(f"   {'Regime 1':<10} {regime_means[1]:<12.4f} {regime_vars[1]:<12.4f} {'High-Rate Environment'}")

print(f"\n   Transition Matrix:")
print(f"   {'-'*40}")
print(f"              To Regime 0    To Regime 1")
print(f"   From 0:       {P[0,0]:.4f}         {P[0,1]:.4f}")
print(f"   From 1:       {P[1,0]:.4f}         {P[1,1]:.4f}")

# 4. Regime Duration Analysis
print("\n[4/5] Regime Duration Analysis...")
duration_0 = 1 / (1 - P[0, 0])
duration_1 = 1 / (1 - P[1, 1])

print(f"   Expected duration in Regime 0 (low rate):  {duration_0:.1f} quarters ({duration_0/4:.1f} years)")
print(f"   Expected duration in Regime 1 (high rate): {duration_1:.1f} quarters ({duration_1/4:.1f} years)")

# Steady-state probabilities
eigenvalues, eigenvectors = eig(P.T)
steady_state = eigenvectors[:, np.argmax(eigenvalues.real)].real
steady_state = steady_state / steady_state.sum()
print(f"\n   Unconditional (long-run) probabilities:")
print(f"   • Regime 0: {steady_state[0]:.2%}")
print(f"   • Regime 1: {steady_state[1]:.2%}")

# 5. Visualization
print("\n[5/5] Generating Regime Probability Plot...")
smoothed_probs = res.smoothed_marginal_probabilities

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Panel A: Federal Funds Rate
axes[0].plot(fed.index, fed.values, color='black', linewidth=1, label='Fed Funds Rate')
axes[0].set_title('Challenge 2A: US Federal Funds Rate (1954-2010)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Rate (%)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Panel B: Smoothed Regime Probabilities
axes[1].fill_between(smoothed_probs.index, 0, smoothed_probs.iloc[:, 0],
                     alpha=0.4, color='blue',
                     label=f'Regime 0: Low Rate (μ={regime_means[0]:.2f}%)')
axes[1].fill_between(smoothed_probs.index, 0, smoothed_probs.iloc[:, 1],
                     alpha=0.4, color='red',
                     label=f'Regime 1: High Rate (μ={regime_means[1]:.2f}%)')
axes[1].set_title('Challenge 2B: Smoothed Regime Probabilities', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Probability')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Annotate key regime shifts
axes[1].axvline(pd.Timestamp('1979-07-01'), color='green', linestyle='--', alpha=0.6, linewidth=2)
axes[1].text(pd.Timestamp('1979-07-01'), 0.5, ' Volcker Era', rotation=90, va='center')

plt.tight_layout()
plt.savefig('challenge2_regime_probabilities.png', dpi=300, bbox_inches='tight')
print("   ✓ Saved: challenge2_regime_probabilities.png")
plt.show()

# ==========================================
# CHALLENGE 3: FEATURE EXTRACTION (PCA)
# ==========================================
print("\n" + "="*70)
print("CHALLENGE 3: PRINCIPAL COMPONENT ANALYSIS (PCA)")
print("="*70)

# 1. Data Collection
print("\n[1/8] Downloading Stock Data...")
# All 10 stocks as specified in report
tickers = ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'JNJ', 'XOM', 'WMT', 'PG', 'DIS', 'BA']
start_date = '2019-01-01'
end_date = '2023-12-31'

try:
    data_stocks = yf.download(tickers, start=start_date, end=end_date, progress=False)['Adj Close']
    spy_data = yf.download('^GSPC', start=start_date, end=end_date, progress=False)['Adj Close']
    print(f"   ✓ Downloaded {len(tickers)} stocks, {len(data_stocks)} days")
except Exception as e:
    print(f"   ✗ Download failed: {e}")
    print("   Please check internet connection and try again.")
    raise

# 2. Feature Engineering
print("\n[2/8] Engineering Technical Indicators...")

def calculate_technical_indicators(prices, returns):
    """
    Construct technical indicators for each stock.

    Returns 4 features per stock:
    - Return: Daily return (momentum signal)
    - MA5: 5-day moving average ratio (short-term trend)
    - MA20: 20-day moving average ratio (long-term trend)
    - BB%: Bollinger Band position (mean-reversion signal)
    """
    features = pd.DataFrame(index=returns.index)

    for ticker in prices.columns:
        price = prices[ticker]

        # 1. Returns
        features[f'{ticker}_Return'] = returns[ticker]

        # 2. MA5 Ratio (price relative to 5-day MA)
        ma5 = price.rolling(window=5).mean()
        features[f'{ticker}_MA5'] = (price / ma5 - 1).reindex(returns.index)

        # 3. MA20 Ratio (price relative to 20-day MA)
        ma20 = price.rolling(window=20).mean()
        features[f'{ticker}_MA20'] = (price / ma20 - 1).reindex(returns.index)

        # 4. Bollinger Band Position
        bb_mid = price.rolling(window=20).mean()
        bb_std = price.rolling(window=20).std()
        bb_lower = bb_mid - 2 * bb_std
        bb_upper = bb_mid + 2 * bb_std
        features[f'{ticker}_BB%'] = ((price - bb_lower) / (bb_upper - bb_lower)).reindex(returns.index)

    return features.dropna()

returns = data_stocks.pct_change().dropna()
spy_returns = spy_data.pct_change().dropna()

X_full = calculate_technical_indicators(data_stocks, returns)
y_full = spy_returns.shift(-1).reindex(X_full.index).dropna()  # Target: Next day SPY return
X_full = X_full.loc[y_full.index]

print(f"   Feature Matrix Shape: {X_full.shape}")
print(f"   ({X_full.shape[0]} days × {X_full.shape[1]} features)")
print(f"   Features per stock: 4 (Return, MA5, MA20, BB%)")

# 3. Diagnostic Tests
print("\n[3/8] Running PCA Diagnostics...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_full)

# Bartlett's Test (tests if correlation matrix is identity)
chi_square, p_value = calculate_bartlett_sphericity(X_scaled)
print(f"   Bartlett's Test of Sphericity:")
print(f"   • Chi-square: {chi_square:.2f}")
print(f"   • p-value: {p_value:.4e} {'(< 0.05 → correlations exist ✓)' if p_value < 0.05 else '(≥ 0.05 → warning!)'}")

# KMO Test (sampling adequacy)
kmo_all, kmo_model = calculate_kmo(X_scaled)
print(f"\n   Kaiser-Meyer-Olkin (KMO) Test:")
print(f"   • Overall KMO: {kmo_model:.4f}")
kmo_rating = "Marvelous" if kmo_model > 0.9 else "Meritorious" if kmo_model > 0.8 else "Middling" if kmo_model > 0.7 else "Mediocre"
print(f"   • Rating: {kmo_rating} (> 0.8 recommended for PCA)")

# 4. Train-Test Split
print("\n[4/8] Splitting Data (80/20 train-test)...")
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full,
    test_size=0.2,
    shuffle=False,  # Preserve temporal order
    random_state=42
)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"   Train: {X_train.shape[0]} days, Test: {X_test.shape[0]} days")

# 5. PCA Execution
print("\n[5/8] Applying Principal Component Analysis...")
pca = PCA()
pca.fit(X_train_scaled)

explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
n_components_90 = np.argmax(cumulative_variance >= 0.90) + 1

print(f"\n   Variance Explained:")
print(f"   • First PC:  {explained_variance[0]:.2%}")
print(f"   • First 5:   {cumulative_variance[4]:.2%}")
print(f"   • First 10:  {cumulative_variance[9]:.2%}")
print(f"   • First 13:  {cumulative_variance[12]:.2%}")
print(f"   • Components for 95%: {n_components_95}")

# 6. Scree Plot
print("\n[6/8] Generating Scree Plot...")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Variance explained by each component
axes[0].bar(range(1, min(21, len(explained_variance)+1)),
            explained_variance[:20], alpha=0.7, color='steelblue')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained')
axes[0].set_title('Challenge 3A: Variance Explained per Component', fontsize=11, fontweight='bold')
axes[0].grid(alpha=0.3, axis='y')

# Panel B: Cumulative variance
components = range(1, len(explained_variance) + 1)
axes[1].plot(components, cumulative_variance, marker='o', linewidth=2, markersize=4)
axes[1].axhline(0.95, color='red', linestyle='--', linewidth=2, label='95% Threshold')
axes[1].axhline(0.90, color='orange', linestyle='--', linewidth=2, label='90% Threshold')
axes[1].axvline(n_components_95, color='red', linestyle='--', alpha=0.5)
axes[1].text(n_components_95 + 0.5, 0.5, f'{n_components_95} PCs', fontsize=10)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance Explained')
axes[1].set_title('Challenge 3B: Cumulative Variance (Scree Plot)', fontsize=11, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_xlim([0, 25])

plt.tight_layout()
plt.savefig('challenge3_scree_plot.png', dpi=300, bbox_inches='tight')
print("   ✓ Saved: challenge3_scree_plot.png")
plt.show()

# 7. Model Comparison
print("\n[7/8] Model Performance Comparison...")
print("   Training Ridge regression models...")

# Model 1: Full Features
model_full = Ridge(alpha=1.0, random_state=42)
model_full.fit(X_train_scaled, y_train)
y_pred_full = model_full.predict(X_test_scaled)

rmse_full = np.sqrt(mean_squared_error(y_test, y_pred_full))
r2_full = r2_score(y_test, y_pred_full)
mae_full = mean_absolute_error(y_test, y_pred_full)

# Model 2: PCA Reduced (10 components)
pca_10 = PCA(n_components=10)
X_train_pca = pca_10.fit_transform(X_train_scaled)
X_test_pca = pca_10.transform(X_test_scaled)

model_pca = Ridge(alpha=1.0, random_state=42)
model_pca.fit(X_train_pca, y_train)
y_pred_pca = model_pca.predict(X_test_pca)

rmse_pca = np.sqrt(mean_squared_error(y_test, y_pred_pca))
r2_pca = r2_score(y_test, y_pred_pca)
mae_pca = mean_absolute_error(y_test, y_pred_pca)

# Directional accuracy
direction_full = np.mean((y_pred_full > 0) == (y_test > 0))
direction_pca = np.mean((y_pred_pca > 0) == (y_test > 0))

print(f"\n   {'Metric':<25} {'Full Features':<18} {'PCA (10 comp.)':<18} {'% Change'}")
print(f"   {'-'*80}")
print(f"   {'RMSE':<25} {rmse_full:<18.6f} {rmse_pca:<18.6f} {(rmse_pca/rmse_full-1)*100:+.2f}%")
print(f"   {'R² Score':<25} {r2_full:<18.6f} {r2_pca:<18.6f} {(r2_pca-r2_full)*100:+.2f} pp")
print(f"   {'MAE':<25} {mae_full:<18.6f} {mae_pca:<18.6f} {(mae_pca/mae_full-1)*100:+.2f}%")
print(f"   {'Directional Accuracy':<25} {direction_full:<18.2%} {direction_pca:<18.2%} {(direction_pca-direction_full)*100:+.2f} pp")
print(f"   {'# Features':<25} {X_train_scaled.shape[1]:<18} {10:<18} {-75.0}%")
print(f"   {'Variance Retained':<25} {'100%':<18} {f'{cumulative_variance[9]:.1%}':<18} {(cumulative_variance[9]-1)*100:.1f} pp")

print(f"\n   Key Finding: PCA reduces dimensionality by {(1-10/X_full.shape[1]):.0%} while")
print(f"   degrading RMSE by only {(rmse_pca/rmse_full-1)*100:.1f}% (acceptable trade-off)")

# 8. Component Interpretation
print("\n[8/8] Principal Component Interpretation...")
print("\n   Top 5 components by economic interpretation:")
print("   " + "-"*70)

component_names = [
    "PC1: Market Factor (systematic risk across all stocks)",
    "PC2: Sector Rotation (Tech vs. Non-Tech divergence)",
    "PC3: Momentum (trend strength and persistence)",
    "PC4: Mean Reversion (overbought/oversold signals)",
    "PC5: Volatility Regime (risk-on vs. risk-off)"
]

for i, name in enumerate(component_names):
    variance_contrib = explained_variance[i]
    print(f"   {name}")
    print(f"      → Explains {variance_contrib:.2%} of variance")

    # Show top 3 feature loadings
    loadings = pca.components_[i]
    top_indices = np.argsort(np.abs(loadings))[-3:][::-1]
    top_features = [X_full.columns[idx] for idx in top_indices]
    top_weights = [loadings[idx] for idx in top_indices]

    print(f"      → Top loadings: ", end="")
    print(", ".join([f"{feat} ({wt:+.3f})" for feat, wt in zip(top_features, top_weights)]))
    print()



IndentationError: unexpected indent (ipython-input-2499513820.py, line 230)